# Module 4.4 - Retrieval-Based Memory

**Reference:** [`04-retrieval-based-memory.md`](./04-retrieval-based-memory.md)

## What you'll do in this notebook

- Store every conversation turn in ChromaDB, keyed by user.
- Retrieve semantically relevant past turns when the user sends a new message.
- Test cross-session recall - end one session, start another, watch the bot pick up where you left off.

## Setup

In [ ]:
import os
import shutil
from datetime import datetime
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set."

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
EMBED_MODEL = "text-embedding-3-small"
MEMORY_PATH = "./conversation_memory_m4_4"

# Start with a clean memory store. Comment this out to keep state between runs.
if os.path.exists(MEMORY_PATH):
    shutil.rmtree(MEMORY_PATH)

memory_client = chromadb.PersistentClient(path=MEMORY_PATH)
print(f"OK: persistent memory at {MEMORY_PATH}")

## Exercise 1 - The memory store

One collection per user. Each turn gets a unique id, its text, an embedding, and metadata (`role`, `turn`, `timestamp`).

In [ ]:
def embed_one(text: str) -> list[float]:
    resp = client.embeddings.create(model=EMBED_MODEL, input=text)
    return resp.data[0].embedding

class ConversationalMemory:
    def __init__(self, user_id: str):
        self.user_id = user_id
        self.collection = memory_client.get_or_create_collection(f"user_{user_id}")
        # Resume turn counter from whatever is already stored.
        self.turn_counter = self.collection.count()
        self.session: list[dict] = []  # in-memory of the CURRENT session only

    def store_turn(self, role: str, content: str) -> None:
        self.turn_counter += 1
        # TODO:
        # self.collection.add(
        #     ids=[f"turn_{self.turn_counter}"],
        #     embeddings=[embed_one(content)],
        #     documents=[content],
        #     metadatas=[{"role": role, "turn": self.turn_counter, "timestamp": datetime.now().isoformat()}],
        # )
        # self.session.append({"role": role, "content": content, "turn": self.turn_counter})
        raise NotImplementedError

    def recall(self, query: str, top_k: int = 3, exclude_recent: int = 4) -> list[dict]:
        """Return past turns relevant to `query`, skipping the last `exclude_recent` turns."""
        if self.collection.count() == 0:
            return []
        # TODO:
        # 1. Embed query.
        # 2. Call self.collection.query(query_embeddings=[...], n_results=top_k,
        #    where={"turn": {"$lt": self.turn_counter - exclude_recent}})
        # 3. Build [{role, content, turn} ...] from the result and return.
        raise NotImplementedError

print("ConversationalMemory skeleton ready")

## Exercise 2 - A chatbot with retrieval memory

On every turn: pull the 3 most relevant past turns, inject them as a system message, append recent session turns, then call the LLM.

In [ ]:
class MemoryChatbot:
    def __init__(self, user_id: str, system_prompt: str):
        self.memory = ConversationalMemory(user_id)
        self.system_prompt = system_prompt

    def chat(self, user_message: str) -> str:
        self.memory.store_turn("user", user_message)
        relevant = self.memory.recall(user_message, top_k=3)

        messages = [{"role": "system", "content": self.system_prompt}]
        if relevant:
            context = "Relevant past conversation:\n" + "\n".join(
                f"[turn {r['turn']}] {r['role']}: {r['content']}" for r in relevant
            )
            messages.append({"role": "system", "content": context})

        # Tail of the current session
        for turn in self.memory.session[-8:]:
            messages.append({"role": turn["role"], "content": turn["content"]})

        reply = client.chat.completions.create(model=MODEL, messages=messages).choices[0].message.content
        self.memory.store_turn("assistant", reply)
        return reply

print("MemoryChatbot skeleton ready")

## Exercise 3 - Cross-session recall

The persistent ChromaDB store means a brand-new `MemoryChatbot` instance for the same user can see everything the previous instance stored.

In [ ]:
# --- Session 1: tell the bot something specific ---
session1 = MemoryChatbot(user_id="alex", system_prompt="You are a helpful assistant with long-term memory.")
print("session 1:", session1.chat("I'm deploying a Flask app and keep hitting a 'permission denied' error on port 80."))
print("session 1:", session1.chat("My stack is Flask + Gunicorn on Ubuntu 22.04."))

print("\n--- pretend the user closes the tab and comes back tomorrow ---\n")

# --- Session 2: new instance, same user_id ---
session2 = MemoryChatbot(user_id="alex", system_prompt="You are a helpful assistant with long-term memory.")
print("session 2:", session2.chat("Hey, what was the deployment problem we were troubleshooting before?"))

The second session should mention the port 80 permission-denied error and (ideally) the Flask/Gunicorn stack, pulled from past turns via retrieval rather than remembered in-process. That's real long-term memory.

## What's next

Recall stores what the user *said*. `05-user-personalisation.ipynb` distils *who the user is* into a persistent profile: technical level, preferences, constraints, goals.